In [41]:
!pip install emoji

In [42]:
#những thư viện cơ bản
import numpy as np
import nltk
import pandas as pd
import matplotlib.pyplot as plt
import re
#kiểm tra emoji
import emoji

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
# from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.naive_bayes import ComplementNB
from sklearn.linear_model import LogisticRegression

import numpy as np
import pandas as pd

STOP_WORDS = None
STEMMER = None
if STOP_WORDS is None:
    nltk.download("stopwords")
    STOP_WORDS = nltk.corpus.stopwords.words("english")

if STEMMER is None:
    STEMMER = nltk.stem.PorterStemmer()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [43]:
vec = TfidfVectorizer(max_df=0.9, max_features=10000, ngram_range=(1, 2),
                      sublinear_tf=True)
model = LogisticRegression(class_weight='balanced')

In [44]:
df = pd.read_parquet('full_df_cleaned.parquet')


In [45]:
X = df.drop('label', axis=1)
y = df['label']

In [46]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2910)

In [47]:
X_train = vec.fit_transform(X_train['text'])
X_test = vec.transform(X_test['text'])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [48]:
print(classification_report(y_test, y_pred))
print(accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.98      0.98      0.98     30502
           1       0.98      0.98      0.98     31966

    accuracy                           0.98     62468
   macro avg       0.98      0.98      0.98     62468
weighted avg       0.98      0.98      0.98     62468

0.9791093039636294
[[29752   750]
 [  555 31411]]


In [49]:
def remove_subject(text):
    return re.sub(r"^\s*subject\s*:\s*", "", text, count=1, flags=re.I)


def clean_html(text):
    # Chuyển nội dung comment và thẻ HTML thành token HTML.
    text = re.sub(r"<!--.*?-->", " HTML ", text, flags=re.S)
    return re.sub(r"<[^>]*>", " HTML ", text)


def clean_email_address(text):
    pattern = r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}"
    return re.sub(pattern, " EMAIL ", text)


def clean_url(text):
    # Nhận diện URL có http(s), www hoặc domain phổ biến.
    pattern = (
        r"(?:https?://|www\.)[^\s<>\"']+"
        r"|(?<![@\w])(?:[a-zA-Z0-9-]+\.)+"
        r"(?:com|net|org|edu|gov|io|co|info|biz|vn)"
        r"(?:/[^\s<>\"']*)?"
    )

    def replace_url(match):
        raw_url = match.group(0)
        url = raw_url.rstrip(".,!?;:)")
        suffix = raw_url[len(url):]
        return f" URL{len(url)} " + suffix

    return re.sub(pattern, replace_url, text, flags=re.I)


def clean_emoji(text):
    # Hỗ trợ emoji gồm nhiều code point nếu phiên bản emoji cho phép.
    try:
        return emoji.replace_emoji(text, replace=" EMOJI ")
    except AttributeError:
        # Tương thích phiên bản emoji cũ.
        return "".join(
            " EMOJI " if emoji.is_emoji(char) else char
            for char in text
        )


def clean_number(text):
    return re.sub(
        r"\d+",
        lambda match: f" NUMBER{len(match.group(0))} ",
        text,
    )


def clean_special_characters(text):
    # Giữ chữ, chữ số và dấu nháy để không phá token URL25/NUMBER3.
    return re.sub(r"[^a-zA-Z0-9\s']", " ", text)


def normalize_word(text):
    return " ".join(STEMMER.stem(word) for word in text.split())


def remove_stopwords(text):
    return " ".join(word for word in text.split() if word not in STOP_WORDS)


def clean_text(text):
    text = str(text).lower()
    text = remove_subject(text)

    # Thay các mẫu đặc biệt trước khi lọc ký tự.
    text = clean_html(text)
    text = clean_email_address(text)
    text = clean_url(text)
    text = clean_emoji(text)
    text = clean_number(text)

    text = clean_special_characters(text)
    text = re.sub(r"\s+", " ", text).strip()
    text = remove_stopwords(text)

    # Chỉ bật stemming nếu đã quyết định dùng nhất quán cho train/test.
    # text = normalize_word(text)

    return text


def label_to_num(label):
    value = str(label).strip().lower()

    if value == "spam":
        return 1
    else:
        return 0


In [50]:
sms = pd.read_csv('spam.csv', encoding='latin-1')
sms = sms.drop(columns=['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'])
sms = sms.rename(columns={'v1': 'label', 'v2': 'text'})

In [51]:
sms['label'] = sms['label'].apply(label_to_num)
sms['text'] = sms['text'].apply(clean_text)

In [52]:
sms.head()

,label,text
0,0,go jurong point crazy available bugis n great ...
1,0,ok lar joking wif u oni
2,1,free entry NUMBER1 wkly comp win fa cup final ...
3,0,u dun say early hor u c already say
4,0,nah think goes usf lives around though


In [53]:
X_spam = sms.drop('label', axis=1)
y_spam = sms['label']

x_spam_train, x_spam_test, y_spam_train, y_spam_test = train_test_split(X_spam, y_spam, test_size=0.2, random_state=2910)
model_sms = LogisticRegression(class_weight='balanced')
vec_sms = TfidfVectorizer(max_df=0.9, max_features=10000, ngram_range=(1, 2),
                      sublinear_tf=True)
x_spam_train = vec_sms.fit_transform(x_spam_train['text'])
x_spam_test = vec_sms.transform(x_spam_test['text'])

model_sms.fit(x_spam_train, y_spam_train)
y_pred = model_sms.predict(x_spam_test)

In [54]:
print(classification_report(y_spam_test, y_pred))
print(accuracy_score(y_spam_test, y_pred))
print(confusion_matrix(y_spam_test, y_pred))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99       962
           1       0.91      0.94      0.93       153

    accuracy                           0.98      1115
   macro avg       0.95      0.96      0.96      1115
weighted avg       0.98      0.98      0.98      1115

0.979372197309417
[[948  14]
 [  9 144]]


In [55]:
import joblib

model_email = joblib.dump(model, 'model_email.pkl')
vec_email = joblib.dump(vec, 'vec_email.pkl')
model_sms = joblib.dump(model_sms, 'model_sms.pkl')
vec_sms = joblib.dump(vec_sms, 'vec_sms.pkl')